# `avh` — five-fold training on frozen AV-HuBERT features

Trains the probe on the frozen token sequences from [`extract_features.ipynb`](extract_features.ipynb),
on the **subject-disjoint** folds, and prints the row for the results board.

| | |
|---|---|
| **Input** | `cached_features/avhubert-video/` + `cached_features/avhubert-audio/` |
| **Cost** | minutes per fold on CPU; the encoder never runs here |
| **Writes** | `preds/cv_avh.csv`, one row per holdout clip |
| **Protocol** | subject-disjoint 5-fold by default; `SPLIT = "official"` for the paper comparison |

The majority baseline on these clips is printed before any model: a row that does not beat
it is not a result. See [`docs/plan.md`](docs/plan.md) for the experiment ladder.

## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
import sys; sys.path.insert(0, '.')
%load_ext autoreload
%autoreload 2

In [ ]:
import copy
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

from lib.dataset import Clips, Protocol, cache_dir
from sklearn.metrics import roc_auc_score
from lib.metrics import best_threshold, evaluate, format_row, majority_baseline
import sys; sys.path.insert(0, 'notebooks/architectures/avhubert')
import model as M

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEVICE)

## 1 — Settings

`VARIANT` is one of `unified`, `video`, `audio`, `late`, `congruence` — the ladder is in
`docs/plan.md`. `unified` is the native read-out over the two token sequences; the others are the
controls that say whether each stream is worth anything.

In [ ]:
DATA = Path("/content/drive/MyDrive/dolos")
DIRECTION = "avh"

# Both streams come from the same AV-HuBERT checkpoint: mouth crops and the speech filterbank
VIDEO_CACHE = "avhubert-video"
AUDIO_CACHE = "avhubert-audio"
SUFFIX = ""                 # "-random" for the smoke-test cache; the numbers mean nothing

VARIANT = "unified"
SPLIT = "subject"           # "subject" (primary) | "official" (paper comparison)
SEED = 0
BATCH_SIZE = 32
EPOCHS = 60
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 5e-2
FOLDS = 5

VIDEO = cache_dir(VIDEO_CACHE + SUFFIX, DATA)
AUDIO = cache_dir(AUDIO_CACHE + SUFFIX, DATA)

for directory in (VIDEO, AUDIO):
    manifest = directory / "manifest.json"
    if manifest.exists():
        meta = json.loads(manifest.read_text())
        print(directory.name, "| weights:", meta.get("settings", {}).get("weights"),
              "| clips:", meta.get("count"))
        if meta.get("settings", {}).get("weights") == "random":
            raise SystemExit(
                "This cache was built on random encoder weights: training on it measures "
                "the encoder, not the probe. Rebuild both caches with the AV-HuBERT checkpoint."
            )

## 2 — Features in RAM

Both caches are small (≈240 MB total), so they are read once into memory and addressed by
clip id. The Drive mount is latency-bound: reading 1,600 small files per epoch would
dominate a 30-second training loop.

In [ ]:
folds = Protocol.load(DATA / "splits")
clips = Clips.load(DATA)
print(clips.describe_source())


class DualFeatures:
    '''Both cached token sequences for one clip: ((video, audio), label).'''

    def __init__(self, clip_ids, videos, audios, clips):
        self.clip_ids = [c for c in clips.keep(clip_ids) if c in videos and c in audios]
        self.videos, self.audios, self.clips = videos, audios, clips

    def __len__(self):
        return len(self.clip_ids)

    def __getitem__(self, index):
        clip_id = self.clip_ids[index]
        return (self.videos[clip_id], self.audios[clip_id],
                torch.tensor(self.clips[clip_id].y, dtype=torch.float32))


def load_cache(directory):
    arrays = {}
    for path in directory.glob("*.npy"):
        arrays[path.stem] = torch.from_numpy(np.load(path).astype(np.float32))
    return arrays


for directory in (VIDEO, AUDIO):
    if not directory.is_dir():
        raise SystemExit(f"no cache at {directory} — run extract_features.ipynb first")

videos, audios = load_cache(VIDEO), load_cache(AUDIO)
print(f"video arrays {len(videos)} | audio arrays {len(audios)}")
video_dim = next(iter(videos.values())).shape[-1]
audio_dim = next(iter(audios.values())).shape[-1]
print(f"video {video_dim}-d | audio {audio_dim}-d")


def fold_parts(fold):
    """(train, val, holdout). The paper's protocol has no validation side."""
    if SPLIT == "official":
        train, test = folds.official(fold)
        return train, None, test
    return folds.subject(fold)


def loader_for(clip_ids, shuffle, seed=SEED):
    dataset = DualFeatures(clip_ids, videos, audios, clips)
    generator = torch.Generator(); generator.manual_seed(seed)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      generator=generator, num_workers=0)


train_ids, val_ids, hold_ids = fold_parts(1)
print(f"fold 1: {len(train_ids)} train / "
      f"{0 if val_ids is None else len(val_ids)} val / {len(hold_ids)} holdout")
print(clips.describe(train_ids))
print(clips.describe(hold_ids))

## 3 — Model

One probe on top of frozen features. `unified` concatenates the two token sequences and
attends over them; the paper's own evaluation does the same. Trainable parameters are
printed because the board asks for them.

In [ ]:
torch.manual_seed(SEED)
net = M.build(VARIANT, video_dim, audio_dim).to(DEVICE)
print(net)
print("trainable:", f"{M.trainable_parameters(net):,}")

## 4 — Training loop

Early stopping on validation **AUC** (thresholds move between folds, AUC does not), the
decision threshold chosen on `val`, `pos_weight` on the training clips so the 56/44 imbalance
of the training side does not become a bias toward `deception`.

In [ ]:
def train_one_fold(fold, seed=SEED, verbose=True):
    train_ids, val_ids, hold_ids = fold_parts(fold)
    dl_train = loader_for(train_ids, True, seed=seed)
    dl_val = loader_for(val_ids, False) if val_ids is not None else None
    dl_hold = loader_for(hold_ids, False)

    torch.manual_seed(seed)
    net = M.build(VARIANT, video_dim, audio_dim).to(DEVICE)
    optimiser = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=clips.class_weights(train_ids)).to(DEVICE)

    best_score, best_state, best_thr, waited = -1.0, None, 0.5, 0
    started = time.perf_counter()
    for epoch in range(EPOCHS):
        net.train()
        for video, audio, y in dl_train:
            video, audio, y = video.to(DEVICE), audio.to(DEVICE), y.to(DEVICE)
            optimiser.zero_grad()
            loss = loss_fn(net(video, audio), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimiser.step()

        if dl_val is None:                       # official split has no validation side
            continue
        net.eval()
        probs, targets = [], []
        with torch.no_grad():
            for video, audio, y in dl_val:
                probs.append(torch.sigmoid(net(video.to(DEVICE), audio.to(DEVICE))).cpu())
                targets.append(y)
        probs = torch.cat(probs).numpy()
        targets = torch.cat(targets).numpy()
        score = roc_auc_score(targets, probs)
        if score > best_score:
            best_score, best_state, waited = score, copy.deepcopy(net.state_dict()), 0
            best_thr = best_threshold(targets, probs)
        else:
            waited += 1
            if waited >= PATIENCE:
                break

    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    probs, targets, ids = [], [], []
    with torch.no_grad():
        for video, audio, y in dl_hold:
            probs.append(torch.sigmoid(net(video.to(DEVICE), audio.to(DEVICE))).cpu())
            targets.append(y)
    probs = torch.cat(probs).numpy()
    targets = torch.cat(targets).numpy()
    metrics = evaluate(targets, probs, threshold=best_thr)
    metrics["fold"] = fold
    metrics["val_auc"] = best_score if dl_val is not None else float("nan")
    metrics["threshold"] = best_thr
    metrics["seconds"] = time.perf_counter() - started
    if verbose:
        validation = f"val AUC {best_score:.3f} | " if dl_val is not None else ""
        print(f"fold {fold}: {validation}holdout ACC {metrics['acc']:.3f} "
              f"AUC {metrics['auc']:.3f} | {metrics['seconds']:.0f}s")
    return metrics, list(hold_ids), targets, probs

## 5 — Five folds

In [ ]:
results, pooled = [], []
for fold in range(1, FOLDS + 1):
    metrics, ids, targets, probs = train_one_fold(fold)
    results.append(metrics)
    pooled.append(pd.DataFrame({"clip_id": ids, "y_true": targets, "y_prob": probs}))

summary = pd.DataFrame(results)[["acc", "f1", "auc", "balanced_acc"]]
print(summary.round(3).to_string())
print(f"\nCV  ACC {summary.acc.mean()*100:.1f} ± {summary.acc.std()*100:.1f}  "
      f"F1 {summary.f1.mean()*100:.1f}  AUC {summary.auc.mean()*100:.1f}")

## 6 — Pooled evaluation, baseline, predictions

The majority baseline is computed on the same clips, not on the full dataset: the board's
reference row uses the working set.

In [ ]:
preds = pd.concat(pooled, ignore_index=True)
overall = evaluate(preds.y_true, preds.y_prob)
base = majority_baseline(preds.y_true)
print(f"pooled ACC {overall['acc']*100:.1f}  balanced {overall['balanced_acc']*100:.1f}  "
      f"AUC {overall['auc']*100:.1f} | majority {base*100:.1f}")

out = Path("preds"); out.mkdir(exist_ok=True)
preds.to_csv(out / f"cv_{DIRECTION}.csv", index=False)
print("saved", out / f"cv_{DIRECTION}.csv")

## 7 — Per gender

In [ ]:
meta = pd.read_csv(DATA / "dataset.csv", dtype=str).set_index("clip_id")
preds["gender"] = preds.clip_id.map(meta.gender)
for gender, group in preds.groupby("gender"):
    m = evaluate(group.y_true, group.y_prob)
    print(f"{gender:7} n={m['n']:4d}  ACC {m['acc']*100:5.1f}  balanced {m['balanced_acc']*100:5.1f}")

## 8 — The board row

Paste it in `README.md`, `## Results board`. `params` is the trainable probe, not the
frozen encoder.

In [ ]:
params = M.trainable_parameters(M.build(VARIANT, video_dim, audio_dim))
runtime = f"{pd.DataFrame(results).seconds.mean()/60:.1f} min/fold"
print(format_row(DIRECTION, {
    "acc": summary.acc.mean(), "acc_std": summary.acc.std(),
    "f1": summary.f1.mean(), "auc": summary.auc.mean(),
    "balanced_acc": summary.balanced_acc.mean(),
}, owner="<your name>", method=f"AV-HuBERT {VARIANT}", params=params,
   majority=base, runtime=runtime))

## 9 — Before you believe the number

- **Is it above the majority baseline, and above `tia`'s audio-only row?** On DOLOS the
  acoustic stream carries most of the signal; a unified probe that does not beat audio-only
  means the visual stream is contributing noise.
- **Is the spread across folds smaller than the gap?** Five folds on ~1,600 clips is a noisy
  estimate; the `±` column is part of the result, not decoration.
- **Does it survive the official protocol?** Run once with `SPLIT = "official"` for the
  comparison with the paper. Do not report that number as the main result: the official
  split is random over clips, so 95–99% of test clips share a speaker with training
  ([`docs/splits.md`](../../../docs/splits.md) and the measurement in [`docs/plan.md`](docs/plan.md)).